# 01 · Building the surface-code circuit

**Goal:** understand what `stim.Circuit.generated` actually produces before we decode anything.

A *rotated surface code* memory experiment encodes one logical qubit, idles it through `rounds` of stabilizer measurement, then measures the logical observable. Every gate, reset, and measurement is noisy (circuit-level noise). We inspect that structure here.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # repo root
sys.path.insert(0, os.path.abspath('.'))
import stim
import numpy as np

## A small, readable circuit
Start with distance 3 so the circuit is small enough to look at. The four noise channels are all set to the same `p` so the threshold sweep later has a single knob.

In [2]:
def build_circuit(distance, rounds, p):
    return stim.Circuit.generated(
        'surface_code:rotated_memory_z',
        distance=distance,
        rounds=rounds,
        after_clifford_depolarization=p,       # 2-qubit gate noise
        after_reset_flip_probability=p,        # ancilla reset error
        before_measure_flip_probability=p,     # measurement error
        before_round_data_depolarization=p,    # data-qubit idling
    )

circuit = build_circuit(distance=3, rounds=3, p=0.005)
print(circuit)

QUBIT_COORDS(1, 1) 1
QUBIT_COORDS(2, 0) 2
QUBIT_COORDS(3, 1) 3
QUBIT_COORDS(5, 1) 5
QUBIT_COORDS(1, 3) 8
QUBIT_COORDS(2, 2) 9
QUBIT_COORDS(3, 3) 10
QUBIT_COORDS(4, 2) 11
QUBIT_COORDS(5, 3) 12
QUBIT_COORDS(6, 2) 13
QUBIT_COORDS(0, 4) 14
QUBIT_COORDS(1, 5) 15
QUBIT_COORDS(2, 4) 16
QUBIT_COORDS(3, 5) 17
QUBIT_COORDS(4, 4) 18
QUBIT_COORDS(5, 5) 19
QUBIT_COORDS(4, 6) 25
R 1 3 5 8 10 12 15 17 19
X_ERROR(0.005) 1 3 5 8 10 12 15 17 19
R 2 9 11 13 14 16 18 25
X_ERROR(0.005) 2 9 11 13 14 16 18 25
TICK
DEPOLARIZE1(0.005) 1 3 5 8 10 12 15 17 19
H 2 11 16 25
DEPOLARIZE1(0.005) 2 11 16 25
TICK
CX 2 3 16 17 11 12 15 14 10 9 19 18
DEPOLARIZE2(0.005) 2 3 16 17 11 12 15 14 10 9 19 18
TICK
CX 2 1 16 15 11 10 8 14 3 9 12 18
DEPOLARIZE2(0.005) 2 1 16 15 11 10 8 14 3 9 12 18
TICK
CX 16 10 11 5 25 19 8 9 17 18 12 13
DEPOLARIZE2(0.005) 16 10 11 5 25 19 8 9 17 18 12 13
TICK
CX 16 8 11 3 25 17 1 9 10 18 5 13
DEPOLARIZE2(0.005) 16 8 11 3 25 17 1 9 10 18 5 13
TICK
H 2 11 16 25
DEPOLARIZE1(0.005) 2 11 16 25
TICK
X

## What to notice
- **Qubit count:** a distance-d rotated code uses d² data qubits plus d²−1 ancilla (measure) qubits.
- **DETECTOR lines:** these are the parity checks the decoder sees — a detector fires when a stabilizer's parity changes between rounds.
- **OBSERVABLE_INCLUDE:** the true logical value we are trying to preserve. A logical error = decoder's prediction disagrees with this.

Count the qubits and detectors to confirm the d² scaling:

In [3]:
print('num qubits   :', circuit.num_qubits)
print('num detectors:', circuit.num_detectors)
print('num observables:', circuit.num_observables)

num qubits   : 26
num detectors: 24
num observables: 1


## Try it
Rebuild at distance 5 and confirm the qubit count jumps to the d² pattern. This is also why physical-qubit overhead in resource estimates scales like ~2d² per logical qubit.

In [4]:
c5 = build_circuit(distance=5, rounds=5, p=0.005)
print('d=5 qubits:', c5.num_qubits, '| detectors:', c5.num_detectors)

d=5 qubits: 64 | detectors: 120
